# Calcul NDWI avec un User Defined Process (UDP)
<br/>
Dans ce notebook, nous allons calculer le NDWI moyen de chaque mois autour de Toulouse.<br/>
Pour cela, nous allons utiliser un process ndwi (une UDP qui sera créée) et aggregator_temporal.<br/>
<br/>
Import et installation des librairies nécessaires

In [ ]:
try:
    import openeo
except:
    !pip install openeo xarray zarr
    import openeo

In [ ]:
# Connexion à l'aggrégateur
con = openeo.connect("https://aggregator.geodes-lab.fr")
con.authenticate_oidc()

## Création de l'UDP ndwi

In [ ]:
# Define the input parameters
from openeo.api.process import Parameter

data = Parameter.datacube("data", description="Input multibands image")
green = Parameter.string("green", description="Label of the green band")
nir = Parameter.string("nir", description="Label of the nir band")

# Define process
from openeo.internal.graph_building import PGNode
from openeo.processes import reduce_dimension

green_node = PGNode(process_id="array_element", arguments={"data": data, "label": green})
nir_node = PGNode(process_id="array_element", arguments={"data": data, "label": nir})
sum_node = PGNode(process_id="add", arguments={"x": green_node, "y": nir_node})
sub_node = PGNode(process_id="subtract", arguments={"x": green_node, "y": nir_node})
div_node = PGNode(process_id="divide", arguments={"x": sub_node, "y": sum_node})

ndwi = reduce_dimension(data=data, dimension="bands", reducer={"process_graph" : div_node})

# Define the result type
res = {"schema": {"type": "object", "subtype": "datacube"}}

# Create process
con.save_user_defined_process(
    user_defined_process_id="ndwi",
    process_graph=ndwi,
    parameters=[data, green, nir],
    returns=res
)

## Création du datacube et du process graph

**Etape 1 :** Utiliser le process "load_collection" pour charger les données correspondant aux bandes, à l'intervalle de temps et à la bbox voulues.

*Note : Les arguments temporal_extent et spatial_extent sont optionels.*

Nous recommandons de filter temporellement, spatialement et sur les bandes directement avec le load collection pour avoir des résultats plus précis.

In [ ]:
df = con.load_collection(
    "S2-L2A-COG", 
    temporal_extent=["2022-01-01","2023-01-01"], 
    spatial_extent={"west": 1.33, "south": 43.53, "east": 1.54, "north": 43.67, "crs": 4326},  # bbox around Toulouse
    bands=["B3", "B8"])
df

**Etape 2 :** Calcul du NDWI avec l'UDP

In [ ]:
df_ndwi = df.process("ndwi", data=df, green="B3", nir="B8")
df_ndwi

**Etape 3 :** Calcul de la moyenne par mois avec une aggrégation temporelle.

In [ ]:
from openeo.processes import mean

df_aggregated = df_ndwi.aggregate_temporal_period(period="month", reducer=mean)
df_aggregated

## Création et lancement du job

In [ ]:
# On veut du Zarr en sortie
job_df = df_aggregated.create_job(title="NDWI Toulouse avec UDP", out_format="Zarr", compress="zstd")

In [ ]:
job_df.start_and_wait()  # environ 7 min

**Une fois que le job est terminé (status=finished), on récupère les résultats**

In [ ]:
# Aperçu des résultats
results = job_df.get_results()
results

In [ ]:
# Téléchargement
results.download_files("./results_ndwi_with_udp/")

## Visualisation des résultats

In [ ]:
import zipfile
import xarray as xr
from pathlib import Path

# Dézipper et ouvrir automatiquement
zip_path = next(Path("./results_ndwi_with_udp/").glob("*.zarr.zip"))
extract_dir = zip_path.parent / zip_path.stem  # retire le .zip

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(extract_dir)

In [ ]:
# Visualisation des résultats
result = xr.open_zarr(extract_dir)
variable_name = list(result.data_vars)[0]

print(result.t.values)

In [ ]:
time = "2022-01-31T00:00:00.000000000"  # vous pouvez tester avec d'autres dates disponibles ci-dessus

result[variable_name].sel(t=time).plot.imshow()

In [ ]:
# Clean

import shutil

shutil.rmtree("./results_ndwi_with_udp/")